# Week 11 Optional: Attention & Transformers Deep Dive

This notebook is a **challenging, optional deep-dive** into the internal mechanics
of transformer models. Work through it at your own pace — after class, over the
weekend, or whenever you want to level up your understanding.

## Prerequisites
- Completed the main Week 11 notebook
- Comfortable with PyTorch basics (Weeks 1-2)
- Watched pre-class videos on attention and transformers

## What You'll Build
1. **Implement scaled dot-product attention** from scratch in NumPy and PyTorch
2. **Build multi-head attention** without using `nn.MultiheadAttention`
3. **Construct a full transformer encoder layer** with positional encoding
4. **Add causal masking** to create a decoder-style attention layer

## Difficulty Level: Challenging

These labs require you to implement core transformer components from scratch.
Use the demos as reference — they show the concepts, then the labs ask you
to build them yourself.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print("\n✅ Ready to explore attention and transformers!")

# Section 1: Dot-Product Attention

### The Problem: Words Change Meaning With Context

Consider the word "bank":
- "I went to the **bank** to deposit money" → financial institution
- "The river **bank** was covered in wildflowers" → edge of a river

![Words change meaning with context](https://www.dropbox.com/s/91xzqre8dpvxrux/sentence.png?raw=1)

**Attention** solves this by looking at ALL surrounding words to determine
the correct meaning of each word in context.

### How Attention Works

![Attention mechanism](https://www.dropbox.com/s/ahn8ogriuzasa9a/attention_in_detail.png?raw=1)

### Query, Key, Value (Q, K, V)

Transformers project each word into three vectors:
- **Query (Q)**: "What am I looking for?"
- **Key (K)**: "What do I contain?"
- **Value (V)**: "What information should I pass along?"

The scaled dot-product attention formula:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

The `√d_k` scaling prevents dot products from getting too large (which would
make softmax produce extreme values near 0 or 1).

In [ ]:
# =============================================================================
# DEMO: Visualizing Attention Weights with NumPy
# =============================================================================
# We'll simulate attention between words using random embeddings.
# In a real transformer, these embeddings are LEARNED during training.

def softmax_np(x, axis=-1):
    """Numerically stable softmax in NumPy."""
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)


def visualize_attention(sentence, title="Attention Weights"):
    """Visualize self-attention heatmap for a sentence."""
    words = sentence.split()
    d_model = 64
    np.random.seed(42)

    # Simulate word embeddings
    embeddings = np.random.randn(len(words), d_model)

    # Q, K projections (simulating learned weight matrices)
    W_q = np.random.randn(d_model, d_model) * 0.1
    W_k = np.random.randn(d_model, d_model) * 0.1

    Q = embeddings @ W_q
    K = embeddings @ W_k

    # Scaled dot-product attention
    scores = Q @ K.T / np.sqrt(d_model)  # Scale by sqrt(d_k)
    weights = softmax_np(scores)           # Normalize to probabilities

    # Plot heatmap
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(weights, cmap='Blues', vmin=0)
    ax.set_xticks(range(len(words)))
    ax.set_yticks(range(len(words)))
    ax.set_xticklabels(words, rotation=45, ha='right', fontsize=10)
    ax.set_yticklabels(words, fontsize=10)
    ax.set_xlabel('Key (attending TO)', fontsize=11)
    ax.set_ylabel('Query (attending FROM)', fontsize=11)
    ax.set_title(title, fontsize=13)
    plt.colorbar(im, ax=ax, label='Attention Weight')
    plt.tight_layout()
    plt.show()
    return weights


# Fraud-related sentence
w = visualize_attention(
    "The customer reported unauthorized charges on credit card",
    "Attention: Fraud Report Sentence"
)

print(f"Each row sums to: {w.sum(axis=-1).round(4)}")
print(f"\n💡 In a TRAINED model, 'unauthorized' would attend strongly to 'charges'.")
print(f"   Our random weights don't show this — training creates meaning!")

## Lab 1: Implement Scaled Dot-Product Attention in PyTorch (Challenging)

### Your Task

Implement the `ScaledDotProductAttention` class as a PyTorch `nn.Module`.
This is the core building block of ALL transformer models.

### Requirements

1. **Create learned projection matrices** W_q, W_k, W_v as `nn.Linear` layers
2. **Compute Q, K, V** by projecting the input through these matrices
3. **Calculate attention scores**: `scores = Q @ K^T / sqrt(d_k)`
4. **Apply optional causal mask** (set future positions to -inf before softmax)
5. **Apply softmax** to get attention weights
6. **Compute context**: `context = weights @ V`
7. **Return** both context vectors and attention weights

### Hints

- `nn.Linear(d_model, d_model, bias=False)` creates a projection matrix
- Use `key.transpose(-2, -1)` to transpose the last two dimensions
- `torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()` creates an upper
  triangular mask (True for positions to mask out)
- `scores.masked_fill_(mask, float('-inf'))` sets masked positions to -inf
- Use `F.softmax(scores, dim=-1)` for softmax

### Expected Output

- Input shape: `(batch, seq_len, d_model)` → Output shape: same
- Attention weights shape: `(batch, seq_len, seq_len)`
- Each row of weights sums to 1.0
- With causal mask: upper triangle of weights should be 0.0

In [ ]:
# =============================================================================
# LAB 1: IMPLEMENT SCALED DOT-PRODUCT ATTENTION
# =============================================================================

class ScaledDotProductAttention(nn.Module):
    """
    Scaled dot-product attention with learned Q, K, V projections.
    Optionally supports causal masking for decoder-style attention.
    """
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model

        # YOUR CODE: Create three linear projection layers (no bias)
        # W_q projects input → queries
        # W_k projects input → keys
        # W_v projects input → values
        self.W_q = None  # YOUR CODE: nn.Linear(d_model, d_model, bias=False)
        self.W_k = None  # YOUR CODE: nn.Linear(d_model, d_model, bias=False)
        self.W_v = None  # YOUR CODE: nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, causal_mask=False):
        """
        Args:
            x: Input tensor of shape (batch, seq_len, d_model)
            causal_mask: If True, apply causal (look-ahead) mask
        Returns:
            context: (batch, seq_len, d_model)
            weights: (batch, seq_len, seq_len)
        """
        # YOUR CODE: Project input to Q, K, V
        Q = None  # YOUR CODE: self.W_q(x)
        K = None  # YOUR CODE: self.W_k(x)
        V = None  # YOUR CODE: self.W_v(x)

        # YOUR CODE: Compute attention scores (Q @ K^T / sqrt(d_k))
        d_k = self.d_model
        scores = None  # YOUR CODE: Q @ K.transpose(-2, -1) / math.sqrt(d_k)

        # YOUR CODE: Apply causal mask if requested
        # Create upper triangular mask and set those positions to -inf
        if causal_mask:
            seq_len = x.size(1)
            mask = None  # YOUR CODE: torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
            # YOUR CODE: scores.masked_fill_(mask, float('-inf'))
            pass  # YOUR CODE: replace this pass with mask application

        # YOUR CODE: Apply softmax to get attention weights
        weights = None  # YOUR CODE: F.softmax(scores, dim=-1)

        # YOUR CODE: Compute context vectors (weights @ V)
        context = None  # YOUR CODE: weights @ V

        return context, weights


# =============================================================================
# TEST YOUR IMPLEMENTATION
# =============================================================================
d_model = 64
batch_size = 2
seq_len = 8

attn = ScaledDotProductAttention(d_model)
x = torch.randn(batch_size, seq_len, d_model)

# Test without mask
context_no_mask = None  # YOUR CODE: call attn(x, causal_mask=False)
weights_no_mask = None  # YOUR CODE: second return value

# Test with causal mask
context_masked = None  # YOUR CODE: call attn(x, causal_mask=True)
weights_masked = None  # YOUR CODE: second return value

# Verification
if context_no_mask is not None and weights_no_mask is not None:
    print(f"✅ Input shape:     {x.shape}")
    print(f"✅ Context shape:   {context_no_mask.shape}  (same as input!)")
    print(f"✅ Weights shape:   {weights_no_mask.shape}")
    print(f"✅ Rows sum to 1:   {weights_no_mask[0].sum(dim=-1).detach().numpy().round(4)}")

    if weights_masked is not None:
        print(f"\n✅ Causal mask applied:")
        print(f"   Upper triangle (should be ~0):")
        upper = torch.triu(weights_masked[0], diagonal=1).detach()
        print(f"   Max value in upper triangle: {upper.max().item():.6f}")
        print(f"\n   Attention weights (first example, causal):")
        print(weights_masked[0].detach().numpy().round(3))
    print(f"\n🎉 Lab 1 complete!")
else:
    print("❌ Lab 1 incomplete — implement ScaledDotProductAttention above")

# Section 2: Multi-Head Attention

Instead of one attention mechanism, transformers use **multiple parallel "heads"**.
Each head learns to focus on **different types of relationships**:

![Multi-Head Attention](https://www.dropbox.com/s/wjfxpap06viclhv/mha.png?raw=1)

| Head | Might Learn | Example |
|------|------------|--------|
| Head 1 | Syntactic relationships | subject ↔ verb agreement |
| Head 2 | Semantic similarity | "charges" ↔ "payment" ↔ "transaction" |
| Head 3 | Positional proximity | nearby words in sentence |
| Head 4 | Long-range dependencies | pronoun ↔ noun it refers to |

**Analogy**: Like CNN filters — each filter detects different visual patterns
(edges, textures, shapes). Each attention head detects different linguistic patterns.

### How It Works

1. Project input to Q, K, V using learned matrices
2. **Split** Q, K, V into `num_heads` smaller chunks (reshape)
3. Run attention **independently** on each chunk (in parallel!)
4. **Concatenate** all head outputs
5. **Project** back to original dimension with output matrix W_o

```python
# The key reshape: (batch, seq, d_model) → (batch, num_heads, seq, d_head)
# where d_head = d_model // num_heads
Q = Q.view(batch, seq, num_heads, d_head).transpose(1, 2)
```

In [ ]:
# =============================================================================
# DEMO: PyTorch's Built-In Multi-Head Attention
# =============================================================================
# This shows what your Lab 2 implementation should match.

seq_len = 6       # 6 tokens in our sequence
batch_size = 2    # 2 examples
d_model = 64      # 64-dimensional embeddings
num_heads = 4     # 4 attention heads (each operates on 64/4 = 16 dims)

# PyTorch MHA expects (seq, batch, d_model) — different from our convention!
x = torch.randn(seq_len, batch_size, d_model)

# Create and run multi-head attention
mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads)
output, attention_weights = mha(query=x, key=x, value=x)

print(f"Input shape:   {x.shape}  → (seq_len, batch, d_model)")
print(f"Output shape:  {output.shape}  → (same as input!)")
print(f"Weights shape: {attention_weights.shape}  → (batch, seq_len, seq_len)")
print(f"\nAttention matrix (averaged across heads, first example):")
print(attention_weights[0].detach().numpy().round(3))
print(f"\nEach row sums to: {attention_weights[0].sum(dim=-1).detach().numpy().round(3)}")
print(f"\n💡 Your Lab 2 goal: build this from scratch!")

## Lab 2: Build Multi-Head Attention from Scratch (Challenging)

### Your Task

Implement `MultiHeadAttention` as an `nn.Module` **without using**
`nn.MultiheadAttention`. You'll implement the full multi-head mechanism:
project → split heads → attend → concatenate → project output.

### Requirements

1. **Create projection matrices** W_q, W_k, W_v (d_model → d_model) and W_o
2. **Reshape** Q, K, V from `(batch, seq, d_model)` to `(batch, num_heads, seq, d_head)`
3. **Compute attention** for each head in parallel (matrix operations handle this)
4. **Concatenate** heads: reshape back from `(batch, num_heads, seq, d_head)` to `(batch, seq, d_model)`
5. **Project output** through W_o

### Key Reshape Operations

```python
# Split heads: (batch, seq, d_model) → (batch, num_heads, seq, d_head)
Q = Q.view(batch, seq, num_heads, d_head).transpose(1, 2)

# Merge heads: (batch, num_heads, seq, d_head) → (batch, seq, d_model)
out = out.transpose(1, 2).contiguous().view(batch, seq, d_model)
```

### Verification

- Output shape should equal input shape
- Attention weights per head: `(batch, num_heads, seq, seq)`
- Each row of each head's weights should sum to 1.0

In [ ]:
# =============================================================================
# LAB 2: BUILD MULTI-HEAD ATTENTION FROM SCRATCH
# =============================================================================

class MultiHeadAttention(nn.Module):
    """
    Multi-head attention implemented from scratch.
    Uses (batch, seq, d_model) convention (not PyTorch's seq-first).
    """
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads  # Dimension per head

        # YOUR CODE: Create projection layers
        self.W_q = None  # YOUR CODE: nn.Linear(d_model, d_model)
        self.W_k = None  # YOUR CODE: nn.Linear(d_model, d_model)
        self.W_v = None  # YOUR CODE: nn.Linear(d_model, d_model)
        self.W_o = None  # YOUR CODE: nn.Linear(d_model, d_model) — output projection

    def split_heads(self, x):
        """Reshape (batch, seq, d_model) → (batch, num_heads, seq, d_head)"""
        batch, seq, _ = x.size()
        # YOUR CODE: reshape and transpose
        result = None  # YOUR CODE: x.view(batch, seq, self.num_heads, self.d_head).transpose(1, 2)
        return result

    def merge_heads(self, x):
        """Reshape (batch, num_heads, seq, d_head) → (batch, seq, d_model)"""
        batch, _, seq, _ = x.size()
        # YOUR CODE: transpose back and reshape
        result = None  # YOUR CODE: x.transpose(1, 2).contiguous().view(batch, seq, self.d_model)
        return result

    def forward(self, x):
        """
        Args:
            x: (batch, seq, d_model)
        Returns:
            output: (batch, seq, d_model)
            attn_weights: (batch, num_heads, seq, seq)
        """
        # Step 1: Project to Q, K, V
        Q = None  # YOUR CODE: self.W_q(x)
        K = None  # YOUR CODE: self.W_k(x)
        V = None  # YOUR CODE: self.W_v(x)

        # Step 2: Split into multiple heads
        Q = None  # YOUR CODE: self.split_heads(Q)
        K = None  # YOUR CODE: self.split_heads(K)
        V = None  # YOUR CODE: self.split_heads(V)

        # Step 3: Compute scaled dot-product attention per head
        scores = None  # YOUR CODE: Q @ K.transpose(-2, -1) / math.sqrt(self.d_head)
        attn_weights = None  # YOUR CODE: F.softmax(scores, dim=-1)
        head_outputs = None  # YOUR CODE: attn_weights @ V

        # Step 4: Concatenate heads
        concat = None  # YOUR CODE: self.merge_heads(head_outputs)

        # Step 5: Final output projection
        output = None  # YOUR CODE: self.W_o(concat)

        return output, attn_weights


# =============================================================================
# TEST YOUR IMPLEMENTATION
# =============================================================================
d_model = 64
num_heads = 4
batch_size = 2
seq_len = 8

mha_custom = MultiHeadAttention(d_model, num_heads)
x = torch.randn(batch_size, seq_len, d_model)

# YOUR CODE: Run forward pass
output = None  # YOUR CODE: first return value of mha_custom(x)
attn_weights = None  # YOUR CODE: second return value

# Verification
if output is not None and attn_weights is not None:
    print(f"✅ Input shape:   {x.shape}")
    print(f"✅ Output shape:  {output.shape}  (should match input)")
    print(f"✅ Weights shape: {attn_weights.shape}  (batch, heads, seq, seq)")
    print(f"✅ Rows sum to 1: {attn_weights[0, 0].sum(dim=-1).detach().numpy().round(4)}")

    # Count parameters
    params = sum(p.numel() for p in mha_custom.parameters())
    print(f"\n   Total parameters: {params:,}")
    print(f"   Expected: 4 × {d_model}×{d_model} = {4 * d_model * d_model + 4 * d_model:,}")
    print(f"\n🎉 Lab 2 complete!")
else:
    print("❌ Lab 2 incomplete — implement MultiHeadAttention above")

# Section 3: The Full Transformer

The transformer combines attention with several other components:

![Transformer Architecture](https://www.tensorflow.org/images/tutorials/transformer/transformer.png)

### Components of a Transformer Layer

![Encoder Layer](https://www.tensorflow.org/images/tutorials/transformer/EncoderLayer.png)

Each layer contains:
1. **Multi-Head Attention** — learn word relationships
2. **Add & Normalize** — residual connection + layer normalization
3. **Feed-Forward Network** — two linear layers + ReLU (adds non-linearity)
4. **Add & Normalize** — another residual connection

### Positional Encoding

Attention is **permutation-invariant** — it doesn't know word ORDER. We add
positional encodings to tell the model where each token is in the sequence:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

### Self-Attention vs Causal Attention

![Self-Attention](https://www.tensorflow.org/images/tutorials/transformer/SelfAttention.png)

**Self-attention** (encoder): Every token sees ALL tokens. For understanding.

![Causal Attention](https://www.tensorflow.org/images/tutorials/transformer/CausalSelfAttention-new-full.png)

**Causal attention** (decoder): Each token only sees PREVIOUS tokens. For generation.

### Three Types of Transformers

| Type | Attention | Examples | Best For |
|------|-----------|----------|----------|
| **Encoder-only** | Bidirectional | BERT, RoBERTa | Classification, NER |
| **Decoder-only** | Causal | GPT-4, Claude, Llama | Text generation |
| **Encoder-Decoder** | Both | T5, BART, Flan-T5 | Translation, summarization |

In [ ]:
# =============================================================================
# DEMO: Reference Implementation — Positional Encoding
# =============================================================================
# Positional encoding uses sin/cos functions at different frequencies
# to encode position information into the embedding space.

def positional_encoding(max_len, d_model):
    """Create sinusoidal positional encoding matrix."""
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(
        torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
    )
    pe[:, 0::2] = torch.sin(position * div_term)  # Even indices: sin
    pe[:, 1::2] = torch.cos(position * div_term)  # Odd indices: cos
    return pe


# Visualize positional encoding
pe = positional_encoding(max_len=50, d_model=64)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full encoding matrix
axes[0].imshow(pe.numpy(), cmap='RdBu', aspect='auto')
axes[0].set_xlabel('Embedding Dimension')
axes[0].set_ylabel('Position')
axes[0].set_title('Positional Encoding Matrix')

# Individual dimensions
for dim in [0, 1, 4, 5, 20, 21]:
    axes[1].plot(pe[:, dim].numpy(), label=f'dim {dim}')
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Value')
axes[1].set_title('Individual Encoding Dimensions')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print("💡 Low dimensions have high frequency (rapid oscillation)")
print("   High dimensions have low frequency (slow variation)")
print("   This lets the model learn both local and global position patterns.")

## Lab 3: Build a Complete Transformer Encoder Layer (Challenging)

### Your Task

Combine everything into a full transformer encoder layer:
1. **Multi-head self-attention** (use PyTorch's `nn.MultiheadAttention` — you
   already built it from scratch in Lab 2!)
2. **Positional encoding** (use the `positional_encoding` function from the demo)
3. **Feed-forward network** (two linear layers with ReLU)
4. **Residual connections + LayerNorm** (Add & Normalize)

Then **stack multiple layers** to create a mini transformer encoder.

### Requirements

**Part A — Single Layer:**
1. Add positional encoding to input embeddings
2. Apply multi-head self-attention + Add & Norm
3. Apply feed-forward network + Add & Norm
4. Return output and attention weights

**Part B — Stacked Encoder:**
1. Create a module that stacks N encoder layers
2. Pass input through all layers sequentially
3. Compare parameter counts for different depths

### Architecture Reference

```
Input Embeddings
    + Positional Encoding
    ↓
┌─────────────────────────────┐
│  Multi-Head Self-Attention  │ ← Layer 1
│  + Residual + LayerNorm     │
│  Feed-Forward Network       │
│  + Residual + LayerNorm     │
└─────────────────────────────┘
    ↓
┌─────────────────────────────┐
│  Multi-Head Self-Attention  │ ← Layer 2
│  + Residual + LayerNorm     │
│  Feed-Forward Network       │
│  + Residual + LayerNorm     │
└─────────────────────────────┘
    ↓
  Output
```

In [ ]:
# =============================================================================
# LAB 3: BUILD A COMPLETE TRANSFORMER ENCODER
# =============================================================================

# --- Part A: Single Encoder Layer ---

class TransformerEncoderLayer(nn.Module):
    """
    One transformer encoder layer:
    Multi-Head Attention → Add & Norm → Feed-Forward → Add & Norm
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()

        # YOUR CODE: Multi-head self-attention
        self.self_attn = None  # YOUR CODE: nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)

        # YOUR CODE: Feed-forward network (Linear → ReLU → Dropout → Linear)
        self.feed_forward = None  # YOUR CODE: nn.Sequential(...)

        # YOUR CODE: Layer normalization (two instances)
        self.norm1 = None  # YOUR CODE: nn.LayerNorm(d_model)
        self.norm2 = None  # YOUR CODE: nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # YOUR CODE: Self-attention with residual connection + layer norm
        # attn_output, attn_weights = self.self_attn(x, x, x)
        # x = self.norm1(x + self.dropout(attn_output))
        attn_output = None  # YOUR CODE
        attn_weights = None  # YOUR CODE

        # YOUR CODE: Feed-forward with residual connection + layer norm
        # ff_output = self.feed_forward(x)
        # x = self.norm2(x + self.dropout(ff_output))

        return x, attn_weights


# --- Part B: Stacked Encoder ---

class TransformerEncoder(nn.Module):
    """
    Full transformer encoder: positional encoding + N stacked layers.
    """
    def __init__(self, d_model, num_heads, d_ff, num_layers, max_len=512, dropout=0.1):
        super().__init__()

        # Positional encoding (precomputed, not learned)
        pe = positional_encoding(max_len, d_model)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

        # YOUR CODE: Create a list of N encoder layers
        self.layers = None  # YOUR CODE: nn.ModuleList([TransformerEncoderLayer(...) for _ in range(num_layers)])

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # YOUR CODE: Add positional encoding
        seq_len = x.size(1)
        # x = self.dropout(x + self.pe[:, :seq_len, :])

        # YOUR CODE: Pass through all layers, collecting attention weights
        all_weights = []
        # for layer in self.layers:
        #     x, weights = layer(x)
        #     all_weights.append(weights)

        return x, all_weights


# =============================================================================
# TEST YOUR IMPLEMENTATION
# =============================================================================
d_model = 64
num_heads = 4
d_ff = 256
num_layers = 4
batch_size = 2
seq_len = 10

# YOUR CODE: Create and test the encoder
encoder = None  # YOUR CODE: TransformerEncoder(d_model, num_heads, d_ff, num_layers)
x = torch.randn(batch_size, seq_len, d_model)

output = None  # YOUR CODE: first return value of encoder(x)
all_weights = None  # YOUR CODE: second return value

# Verification
if output is not None and all_weights is not None:
    print(f"✅ Input shape:    {x.shape}")
    print(f"✅ Output shape:   {output.shape}  (same as input!)")
    print(f"✅ Num layers:     {len(all_weights)}")

    # Count parameters
    total_params = sum(p.numel() for p in encoder.parameters())
    print(f"\n   Parameters per layer: ~{total_params // num_layers:,}")
    print(f"   Total parameters:     {total_params:,}")

    # Compare to real models
    print(f"\n📊 Scale comparison:")
    print(f"   Our mini encoder:  {num_layers} layers, {total_params:,} params")
    print(f"   BERT-base:         12 layers, 110M params")
    print(f"   GPT-2 (small):     12 layers, 124M params")
    print(f"   GPT-3:             96 layers, 175B params")
    print(f"\n🎉 Lab 3 complete! You've built a transformer from scratch!")
else:
    print("❌ Lab 3 incomplete — implement TransformerEncoderLayer and TransformerEncoder above")

# Great Work!

If you completed all three labs, you've built the core components of a transformer
**entirely from scratch**:

✅ Scaled dot-product attention with causal masking
✅ Multi-head attention with head splitting and merging
✅ Full transformer encoder with positional encoding

## How This Connects to the Course

| Week | Topic | Connection |
|------|-------|--------------------------------------|
| **13** | Amazon Bedrock | Managed access to these transformer models |
| **14** | Training AI Models (LoRA/QLoRA) | Modify attention layer weights for your domain |
| **15-16** | Agentic AI | Orchestrating transformer-based reasoning |
| **17-18** | RAG | Attention over retrieved documents + query |

## Bonus Challenges (if you want more)

1. **Add causal masking to Lab 3** — modify the encoder layer to support both
   encoder-style (bidirectional) and decoder-style (causal) attention
2. **Implement learned positional embeddings** — replace sin/cos with `nn.Embedding`
3. **Build a mini language model** — add a linear head on top of your encoder
   that predicts the next token from a vocabulary
4. **Visualize attention across layers** — plot the attention weights from each
   layer side-by-side. How do they differ?

## Further Reading

- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/)
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762) — the original paper
- [The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/) — Harvard NLP
- [3Blue1Brown: Attention in Transformers](https://www.youtube.com/watch?v=eMlx5fFNoYc)